In [1]:
import sys
import os
import time
import uuid
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Setup context
from dotenv import load_dotenv
load_dotenv()
from core import enable_logging
enable_logging()
# 将项目根目录加入模块路径
from agent.BasicAgent import BasicAgent
from core.llm import EasyLLM
from skill.registry import SkillRegistry
from skill.builtin.calculator_skill import CalculatorSkill
from skill.yaml_loader import YAMLSkillLoader, MarkdownSkillLoader
from skill.folder_loader import FolderSkillLoader
from skill import MetaSkill
from pydantic import BaseModel,Field
from Tool import Tool
from skill import BaseSkill
from skill import SkillConfig
class TranslateParams(BaseModel):
    text: str = Field(description="要翻译的文本")
    target_lang: str = Field(default="en", description="目标语言")

class TranslateTool(Tool):
    def __init__(self):
        super().__init__("translate_tool", "将文本翻译为目标语言", TranslateParams)

    def run(self, parameters: dict) -> str:
        # 实际翻译逻辑
        return f"Translated: {parameters['text']}"

# 2. 定义 Skill
class TranslateSkill(BaseSkill):
    def __init__(self):
        config = SkillConfig(
            name="translate",
            description="多语言翻译技能",
            version="1.0.0",
            tags=["translate", "language", "i18n"],
            priority=5,
        )
        super().__init__(config)

    def get_tools(self) -> list:
        return [TranslateTool()]

    def get_prompt(self) -> str:
        return """## 翻译能力
你具备多语言翻译能力。当用户要求翻译时，请使用 translate_tool 工具。
- 支持中英日韩等多种语言
- 可以自动识别源语言
"""

def test_invoke_without_tool(agent):

    agent.clear_history()

    result=agent.invoke("你好，请介绍一下你自己")
    print(result)

async def test_ainvoke_without_tool(agent):
    agent.clear_history()


    result=await agent.ainvoke("你好，请介绍一下你自己")
    print(result)

def test_stream_without_tool(agent):
    agent.clear_history()

    agent.stream_invoke("你好，请介绍一下你自己")

async def test_astream_without_tool(agent):
    agent.clear_history()

    await agent.astream_invoke("你好，请介绍一下你自己")

def test_invoke_with_tool(agent):
    agent.clear_history()
    result=agent.invoke(f"使用工具计算3^12 ")
    # result=agent.invoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")
    print(result)

async def test_ainvoke_with_tool(agent):
    agent.clear_history()


    result=await agent.ainvoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")
    print(result)

def test_stream_with_tool(agent):
    agent.clear_history()


    agent.stream_invoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")

async def test_astream_with_tool(agent):
    agent.clear_history()

    await agent.astream_invoke(f"使用工具计算3^12 ")
    # await agent.astream_invoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")



In [2]:
llm= EasyLLM(provider="google_native")
from context import ContextManager,LLMHistoryCompactor
# context_manage=ContextManager()
# context_manage.set_history_compactor(LLMHistoryCompactor(llm=llm))
agent=BasicAgent(name="test_skill", llm=llm,reasoning={"effort":"high"},verbose_thinking=True)
# agent.with_context(context_manage)

2026-04-23 04:27:31,520 | INFO | EasyLLM 初始化完成: provider=google_native, model=gemini-3-flash
2026-04-23 04:27:31,664 | INFO | BasicAgent 'test_skill' 初始化完成，工具调用: 禁用，provider: google_native


In [3]:
agent.with_skill(CalculatorSkill())
agent.with_skill(TranslateSkill())

2026-04-23 04:27:32,333 | INFO | 📦 注册 Skill 'calculator' (v1.0.0)
2026-04-23 04:27:32,333 | INFO | ✅ 激活 Skill 'calculator' (工具: ['calculator'])
2026-04-23 04:27:32,334 | INFO | 📦 注册 Skill 'translate' (v1.0.0)
2026-04-23 04:27:32,334 | INFO | ✅ 激活 Skill 'translate' (工具: ['translate_tool'])


In [30]:
await test_astream_with_tool(agent)


2026-04-23 04:39:30,313 | INFO | 对话历史已清空


round 1

thinking content:
**Commencing the Calculation**

I'm starting the process of calculating $3^{12}$. The user's request is straightforward, and the calculator tool seems like the most direct approach. I'm focusing on leveraging the tool to get the numerical result efficiently.



tool_calls:
calculator : {'expression': '3**12'}

round 2

content:
3 的 12 次方等于 531,441。
final res:
3 的 12 次方等于 531,441。


In [31]:
agent.get_raw_history()


[{'role': 'user', 'parts': [{'text': '使用工具计算3^12 '}]},
 {'role': 'model',
  'parts': [{'text': "**Commencing the Calculation**\n\nI'm starting the process of calculating $3^{12}$. The user's request is straightforward, and the calculator tool seems like the most direct approach. I'm focusing on leveraging the tool to get the numerical result efficiently.\n\n\n",
    'thought': True},
   {'function_call': {'id': None,
     'name': 'calculator',
     'args': {'expression': '3**12'}},
    'thought_signature': 'CiQBjz1rX1ETyy1WhGRcEn81TRhP2CWT6/n39tRqLm4/GstRqjYKZwGPPWtf2EZjWtI9ooVVttPcil7x6lz249ViDL3ELfl4/wvIZFXxZ4zuYv6OWLUMXVEfduA5hwO/fDB30ovKQHd1GOlNKE7EYvU3pYpkTXTF7q2CZwgfmHS6OH0GH9hK59UY4BMuz/wKOwGPPWtfc/yTubMmdbuCzmTKL6wSX6Tzo6kGGgqoYULUGW4sboLlkMlHPu7zOJIjTQB3MouCLdogEv6K'}]},
 {'role': 'user',
  'parts': [{'function_response': {'name': 'calculator',
     'response': {'result': '531441'}}}]},
 {'role': 'model', 'parts': [{'text': '3 的'}, {'text': ' 12 次方等于 531,441。'}]}]

In [29]:
agent.get_raw_history() #wrong

[{'role': 'user', 'parts': [{'text': '使用工具计算3^12 '}]},
 {'role': 'model',
  'parts': [{'function_call': {'id': 'qvxw219s',
     'name': 'calculator',
     'args': {'expression': '3**12'}},
    'thought_signature': 'EjQKMgEMOdbHQFECk1k7jnMs2u56XV+tgmOMXSHd6YjGmmunAO82LdnRVGfPTg04dDcpoWrA'}]},
 {'role': 'user',
  'parts': [{'function_response': {'name': 'calculator',
     'response': {'result': '531441'},
     'id': 'qvxw219s'}}]}]

In [ ]:
test_invoke_without_tool(agent)

In [ ]:
agent.get_canonical_history()

In [ ]:
await test_ainvoke_without_tool(agent)

In [ ]:
test_stream_without_tool(agent)

In [ ]:
await test_astream_without_tool(agent)

In [ ]:
agent.with_skill(CalculatorSkill())
agent.with_skill(TranslateSkill())

In [ ]:
test_invoke_with_tool(agent)

In [ ]:
await test_ainvoke_with_tool(agent)

In [ ]:
test_stream_with_tool(agent)

In [ ]:
await test_astream_with_tool(agent)

In [ ]:
raw_history=agent.get_raw_history()  
raw_history

In [ ]:
new_history=raw_history[:-1]
new_history[1]['parts']=new_history[1]['parts'][1:]

In [ ]:
new_history

In [ ]:
agent.llm.invoke_raw(new_history)

In [ ]:
llm= EasyLLM(provider="google",base_url="http://210.45.70.84:30000/v1")
agent.change_model(llm=llm)

In [ ]:
raw_history2=agent.get_raw_history()  

In [ ]:
raw_history2

In [ ]:

await agent.astream_invoke(f"我们刚才聊了什么")


In [ ]:
llm= EasyLLM(provider="openai",base_url="http://127.0.0.1:5124/v1",api_key="122",model="qwen3.5-9b")

agent.change_model(llm=llm)

In [ ]:
raw_history3=agent.get_raw_history()  
raw_history==raw_history3

In [ ]:
await agent.astream_invoke(f"我们刚才聊了什么")
